In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
# Gold Fact Table: fact_bookings
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.fact_bookings AS
SELECT
    fb_bookingid          AS booking_id,
    fb_linenumber         AS booking_line,
    fb_routeid            AS route_id,
    fb_airlineid          AS airline_id,
    fb_seats              AS seats,
    fb_ticketprice        AS ticket_price,
    fb_discount           AS discount_pct,
    fb_tax                AS tax_pct,
    fb_refundflag         AS refund_flag,
    fb_segmentstatus      AS segment_status,
    fb_departuredate      AS departure_date,
    fb_commitdate         AS commit_date,
    fb_arrivaldate        AS arrival_date,
    fb_cabinclass         AS cabin_class,
    fb_comment            AS booking_comment
FROM {catalog}.silver.flight_booking
""")

In [0]:
# Gold Dimension Table: dim_passenger
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_passenger AS
SELECT
    pa_passengerid AS passenger_id,
    pa_name        AS name,
    pa_address     AS address,
    pa_nationkey   AS nation_key,
    pa_phone       AS phone,
    pa_loyaltypoints AS loyalty_points,
    pa_travelpurpose AS travel_purpose,
    pa_comment     AS comment
FROM {catalog}.silver.passenger
""")

In [0]:
# Gold Dimension Table: dim_route
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_route AS
SELECT
    ro_routeid         AS route_id,
    ro_name            AS name,
    ro_aircraft        AS aircraft,
    ro_airline_brand   AS airline_brand,
    ro_routetype       AS route_type,
    ro_capacity        AS capacity,
    ro_cabinconfig     AS cabin_config,
    ro_baseprice       AS base_price,
    ro_comment         AS comment
FROM {catalog}.silver.route
""")

In [0]:
# Gold Dimension Table: dim_airline
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_airline AS
SELECT
    al_airlineid AS airline_id,
    al_name      AS name,
    al_address   AS address,
    al_nationkey AS nation_key,
    al_phone     AS phone,
    al_rating    AS rating,
    al_comment   AS comment
FROM {catalog}.silver.airline
""")

# Gold Dimension Table: dim_geography
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_geography AS
SELECT
    n.n_nationkey    AS nation_id,
    n.n_name         AS nation_name,
    n.n_comment      AS nation_comment,
    r.r_regionkey    AS region_id,
    r.r_name         AS region_name,
    r.r_comment      AS region_comment
FROM {catalog}.silver.nation n
JOIN {catalog}.silver.region r
  ON n.n_regionkey = r.r_regionkey
""")

# Gold KPI View: vw_annual_booking_kpis
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_booking_kpis AS
SELECT
    year(fb_departuredate) AS booking_year,
    SUM(fb_ticketprice * (1 - fb_discount) + fb_tax) AS net_revenue,
    COUNT(DISTINCT fb_bookingid) AS distinct_bookings,
    COUNT(DISTINCT b.bk_passengerid) AS active_passengers,
    AVG(fb_ticketprice) AS avg_booking_value
FROM {catalog}.silver.flight_booking fb
LEFT JOIN {catalog}.silver.booking b
  ON fb.fb_bookingid = b.bk_bookingid
GROUP BY booking_year
ORDER BY booking_year
""")

In [0]:
# Gold/Silver Reconciliation: Revenue, Row Count, and Uniqueness Validation

gold_rev = spark.table(f"{catalog}.gold.fact_bookings").selectExpr("SUM(ticket_price * (1 - discount_pct) + tax_pct) AS revenue").collect()[0][0]
gold_rows = spark.table(f"{catalog}.gold.fact_bookings").count()

silver_rev = spark.table(f"{catalog}.silver.flight_booking").selectExpr("SUM(fb_ticketprice * (1 - fb_discount) + fb_tax) AS revenue").collect()[0][0]
silver_rows = spark.table(f"{catalog}.silver.flight_booking").count()

print(f"Gold revenue:  {gold_rev}")
print(f"Silver revenue:{silver_rev}")
print(f"Gold row count:  {gold_rows}")
print(f"Silver row count:{silver_rows}")

rev_diff = abs(gold_rev - silver_rev)
rows_diff = abs(gold_rows - silver_rows)

print("PASS" if rev_diff < 1e-6 else "FAIL: Revenue mismatch!", f"(Δ={rev_diff})")
print("PASS" if rows_diff == 0 else f"FAIL: Row count mismatch! (Δ={rows_diff})")

# Validate uniqueness of dimension keys
def check_unique(df, key, dim_name):
    dup = df.groupBy(key).count().filter("count > 1")
    num_dup = dup.count()
    if num_dup == 0:
        print(f"PASS: {dim_name} keys are unique")
    else:
        print(f"FAIL: {dim_name} has {num_dup} duplicate keys!")
        display(dup)

for dim, key in [
    (f"{catalog}.gold.dim_passenger", "passenger_id"),
    (f"{catalog}.gold.dim_route", "route_id"),
    (f"{catalog}.gold.dim_airline", "airline_id"),
    (f"{catalog}.gold.dim_geography", "nation_id")
]:
    check_unique(spark.table(dim), key, dim)


In [0]:
# Save reconciliation result as gold.reconciliation_summary table
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, BooleanType

# Calculate reconciliation metrics again

gold_revenue = float(spark.table(f"{catalog}.gold.fact_bookings").selectExpr("SUM(ticket_price * (1 - discount_pct) + tax_pct) AS revenue").collect()[0][0])
silver_revenue = float(spark.table(f"{catalog}.silver.flight_booking").selectExpr("SUM(fb_ticketprice * (1 - fb_discount) + fb_tax) AS revenue").collect()[0][0])
gold_row_count = int(spark.table(f"{catalog}.gold.fact_bookings").count())
silver_row_count = int(spark.table(f"{catalog}.silver.flight_booking").count())
revenue_diff = gold_revenue - silver_revenue
row_diff = gold_row_count - silver_row_count
revenue_pass = abs(revenue_diff) < 1e-6
row_pass = gold_row_count == silver_row_count

schema = StructType([
    StructField("gold_revenue", DoubleType(), False),
    StructField("silver_revenue", DoubleType(), False),
    StructField("gold_row_count", IntegerType(), False),
    StructField("silver_row_count", IntegerType(), False),
    StructField("revenue_diff", DoubleType(), False),
    StructField("row_diff", IntegerType(), False),
    StructField("revenue_pass", BooleanType(), False),
    StructField("row_pass", BooleanType(), False),
])
data = [(gold_revenue, silver_revenue, gold_row_count, silver_row_count, revenue_diff, row_diff, revenue_pass, row_pass)]
rec_df = spark.createDataFrame(data, schema=schema)
rec_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.reconciliation_summary")

print(f"Reconciliation summary written to table {catalog}.gold.reconciliation_summary")
